# Qwen2-VL Hard Negative Permutation Reranker

This notebook tests a hard-negative permutation reranker on top of the existing best Qwen2-VL adapter.

Reference adapter:

```text
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter
```

Experiment:

1. Use the existing `pairwise + first + last` structured decoder to score all 24 permutations.
2. Cache top-K candidates for train / quick50 / tuning150 / holdout150 / test.
3. Build A/B hard-negative records where exactly one side is the gold order.
4. Continue training the existing adapter with ranking-only A/B examples.
5. Evaluate reranker tournament selection against the structured baseline.


In [ ]:
# 1) Install dependencies, then restart runtime once.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_qwen2vl_reranker_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")


In [ ]:
# 2) Setup
from google.colab import drive
drive.mount("/content/drive")

import ast
import gc
import hashlib
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
from peft import PeftModel, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
warnings.filterwarnings("ignore", message=".*The following generation flags are not valid.*")
transformers_logging.set_verbosity_error()

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct"
INITIAL_ADAPTER_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter"
SPLIT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/id_splits/qwen2vl_lgt_order_refine_20260714_003635"

OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_hard_negative_permutation_reranker_v1"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(OUTPUT_ROOT, "runs", RUN_ID)
OUTPUT_DIR = os.path.join(RUN_ROOT, "reranker")
CACHE_DIR = os.path.join(OUTPUT_DIR, "structured_candidate_cache")
RECORD_DIR = os.path.join(OUTPUT_DIR, "reranker_records")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_adapter")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
RUN_SPLIT_DIR = os.path.join(RUN_ROOT, "splits")

SEED = 42
VALID_RATIO = 0.10
QUICK_EVAL_ROWS = 50
TUNING_EVAL_ROWS = 150
HOLDOUT_EVAL_ROWS = 150

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

STRUCTURED_ALPHA = 1.0
STRUCTURED_BETA = 1.0
STRUCTURED_GAMMA = 1.0
CANDIDATE_TOP_K = 10
RERANK_TOP_K_GRID = [3, 5, 10]
LAMBDAS = [0.0, 0.25, 0.5, 0.75, 1.0]

NEGATIVE_RATIOS = {
    "decoder_top_wrong": 0.40,
    "first_error": 0.20,
    "last_error": 0.20,
    "middle_swap": 0.15,
    "random": 0.05,
}
MAX_RECORDS_PER_SAMPLE = 4

LEARNING_RATE = 3e-6
MAX_TRAIN_STEPS = 800
SAVE_STEPS = 200
LOGGING_STEPS = 20
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4

for directory in [OUTPUT_ROOT, RUN_ROOT, OUTPUT_DIR, CACHE_DIR, RECORD_DIR, EVAL_DIR, BEST_ADAPTER_DIR, RUN_SPLIT_DIR, SPLIT_DIR]:
    os.makedirs(directory, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(INITIAL_ADAPTER_DIR, "adapter_config.json")), INITIAL_ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("output:", OUTPUT_DIR)
print("initial adapter:", INITIAL_ADAPTER_DIR)


In [ ]:
# 3) IO, split, and data helpers
def save_json(data, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(records, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def rows_by_ids(dataframe, ids):
    ids = [str(value) for value in ids]
    subset = dataframe[dataframe["Id"].isin(ids)].copy()
    order = {sample_id: index for index, sample_id in enumerate(ids)}
    subset["_split_order"] = subset["Id"].map(order)
    return subset.sort_values("_split_order").drop(columns=["_split_order"]).reset_index(drop=True)


def assert_unique_ids(name, ids):
    ids = [str(value) for value in ids]
    duplicated = pd.Series(ids).value_counts()
    duplicated = duplicated[duplicated > 1]
    assert duplicated.empty, f"{name} has duplicated Ids: {duplicated.head().to_dict()}"


def validate_split_ids(split_ids, all_ids):
    all_ids = set(str(value) for value in all_ids)
    for name, ids in split_ids.items():
        ids = [str(value) for value in ids]
        assert_unique_ids(name, ids)
        missing = sorted(set(ids) - all_ids)
        assert not missing, f"{name} has Ids not found in train.csv: {missing[:5]}"

    train_ids = set(split_ids["train_ids.json"])
    validation_ids = set(split_ids["validation_ids.json"])
    quick_ids = set(split_ids["quick50_ids.json"])
    tuning_ids = set(split_ids["tuning150_ids.json"])
    holdout_ids = set(split_ids["holdout150_ids.json"])
    assert train_ids.isdisjoint(validation_ids), "train_ids and validation_ids overlap"
    assert quick_ids <= validation_ids, "quick50_ids must be a subset of validation_ids"
    assert tuning_ids <= validation_ids, "tuning150_ids must be a subset of validation_ids"
    assert holdout_ids <= validation_ids, "holdout150_ids must be a subset of validation_ids"
    assert quick_ids.isdisjoint(tuning_ids), "quick50_ids and tuning150_ids overlap"
    assert quick_ids.isdisjoint(holdout_ids), "quick50_ids and holdout150_ids overlap"
    assert tuning_ids.isdisjoint(holdout_ids), "tuning150_ids and holdout150_ids overlap"


def derive_eval_split_ids(validation_ids):
    validation_shuffle = [str(value) for value in validation_ids]
    rng = np.random.default_rng(SEED)
    rng.shuffle(validation_shuffle)
    quick50_ids = validation_shuffle[:min(QUICK_EVAL_ROWS, len(validation_shuffle))]
    remaining = validation_shuffle[len(quick50_ids):]
    tuning150_ids = remaining[:min(TUNING_EVAL_ROWS, len(remaining))]
    holdout150_ids = remaining[len(tuning150_ids):len(tuning150_ids) + min(HOLDOUT_EVAL_ROWS, max(0, len(remaining) - len(tuning150_ids)))]
    return quick50_ids, tuning150_ids, holdout150_ids


def make_or_load_split_ids(dataframe):
    names = ["train_ids.json", "validation_ids.json", "quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]
    paths = {name: os.path.join(SPLIT_DIR, name) for name in names}
    split_ids = {}

    if os.path.exists(paths["train_ids.json"]) and os.path.exists(paths["validation_ids.json"]):
        split_ids["train_ids.json"] = [str(value) for value in load_json(paths["train_ids.json"])]
        split_ids["validation_ids.json"] = [str(value) for value in load_json(paths["validation_ids.json"])]
        print("Loaded fixed train/validation split:", SPLIT_DIR)
    else:
        unique_ids = dataframe["Id"].unique().copy()
        rng = np.random.default_rng(SEED)
        rng.shuffle(unique_ids)
        valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
        split_ids["validation_ids.json"] = [str(value) for value in unique_ids[:valid_size]]
        split_ids["train_ids.json"] = [str(value) for value in unique_ids[valid_size:]]
        save_json(split_ids["train_ids.json"], paths["train_ids.json"])
        save_json(split_ids["validation_ids.json"], paths["validation_ids.json"])
        print("Created fixed train/validation split:", SPLIT_DIR)

    quick_ids, tuning_ids, holdout_ids = derive_eval_split_ids(split_ids["validation_ids.json"])
    derived = {
        "quick50_ids.json": quick_ids,
        "tuning150_ids.json": tuning_ids,
        "holdout150_ids.json": holdout_ids,
    }
    for name in ["quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]:
        if os.path.exists(paths[name]):
            split_ids[name] = [str(value) for value in load_json(paths[name])]
        else:
            split_ids[name] = derived[name]
            save_json(split_ids[name], paths[name])
            print("Created missing eval split:", paths[name])

    validate_split_ids(split_ids, dataframe["Id"].astype(str).tolist())
    for name, ids in split_ids.items():
        save_json(ids, os.path.join(RUN_SPLIT_DIR, name))
    return split_ids


def ids_hash(ids):
    text = "\n".join(str(value) for value in ids)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV) if os.path.exists(SAMPLE_SUBMISSION_CSV) else None
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_ids = make_or_load_split_ids(train_df)
training_df = rows_by_ids(train_df, split_ids["train_ids.json"])
validation_df = rows_by_ids(train_df, split_ids["validation_ids.json"])
quick50_df = rows_by_ids(train_df, split_ids["quick50_ids.json"])
tuning150_df = rows_by_ids(train_df, split_ids["tuning150_ids.json"])
holdout150_df = rows_by_ids(train_df, split_ids["holdout150_ids.json"])

print("train/validation/quick/tuning/holdout:", len(training_df), len(validation_df), len(quick50_df), len(tuning150_df), len(holdout150_df))
assert len(training_df) == 8582, f"train split mismatch: {len(training_df)} != 8582"
assert len(validation_df) == 953, f"validation split mismatch: {len(validation_df)} != 953"


In [ ]:
# 4) Model loading and existing structured decoder prompts
def ensure_base_model_path():
    if os.path.exists(os.path.join(DRIVE_MODEL_DIR, "config.json")):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    if not USE_MODELSCOPE_BASE_MODEL:
        print("Using Hugging Face repo id:", MODEL_REPO_ID)
        return MODEL_REPO_ID
    print("Base model cache not found. Downloading via ModelScope:", DRIVE_MODEL_DIR)
    from modelscope import snapshot_download as modelscope_snapshot_download
    model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if not os.path.exists(DRIVE_MODEL_DIR):
        shutil.copytree(model_dir, DRIVE_MODEL_DIR)
    return DRIVE_MODEL_DIR


def load_model_class():
    if Qwen2VLForConditionalGeneration is not None:
        return Qwen2VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError("No compatible Qwen2-VL model class found. Restart runtime after dependency install.")


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def model_device(active_model):
    return next(active_model.parameters()).device


def sanitize_generation_config(active_model):
    if hasattr(active_model, "generation_config"):
        active_model.generation_config.do_sample = False
        active_model.generation_config.temperature = None
        active_model.generation_config.top_p = None
        active_model.generation_config.top_k = None
        active_model.generation_config.num_beams = 1
    return active_model


def load_base_model(for_training=False):
    model_cls = load_model_class()
    model = model_cls.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=MODEL_LOCAL_FILES_ONLY,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    if for_training:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    return model


def load_adapter_model(adapter_dir, is_trainable=False):
    base = load_base_model(for_training=is_trainable)
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=is_trainable)
    model.config.use_cache = False
    sanitize_generation_config(model)
    if not is_trainable:
        model.eval()
    return model


def original_task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nQuestion: Which image represents the beginning of the story?\nAnswer only the image number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nQuestion: Which image represents the end of the story?\nAnswer only the image number from 1 to 4."
    raise ValueError(task_type)


def original_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + original_task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


def make_original_eval_example(row, task_type, image_root, pair=None):
    image_paths = row_image_paths(row, image_root)
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{value!r} tokenized to {ids}; this notebook expects single-token labels.")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}
AB_TOKEN_IDS = {"A": single_token_id("A"), "B": single_token_id("B")}
print("digit token ids:", DIGIT_TOKEN_IDS)
print("A/B token ids:", AB_TOKEN_IDS)
for value in ["A", "B", " A", " B"]:
    print(value, processor.tokenizer.encode(value, add_special_tokens=False))


In [ ]:
# 5) Candidate generation from the existing structured decoder
@torch.no_grad()
def score_digit_candidates(active_model, example, candidates):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "right"
        text = processor.apply_chat_template(original_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example["image_paths"]]
        inputs = processor(text=[text], images=[images], return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
        logits = outputs.logits[0, last_pos]
        token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
        probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
        return {int(candidate): float(prob) for candidate, prob in zip(candidates, probs)}
    finally:
        processor.tokenizer.padding_side = old_padding_side


def structured_score_from_probs(first_probs, last_probs, pair_probs, order, alpha, beta, gamma):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(pair_probs[f"{order[i]}>{order[j]}"]) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(first_probs[str(order[0])]) + eps)
    last_score = math.log(float(last_probs[str(order[-1])]) + eps)
    return float(alpha * pair_score + beta * first_score + gamma * last_score)


def order_metric_row(pred_order, gold_order):
    pred_ranks = {int(image_number): position for position, image_number in enumerate(pred_order)}
    gold_ranks = {int(image_number): position for position, image_number in enumerate(gold_order)}
    pair_accuracy = np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])
    return {
        "exact_match": float(list(pred_order) == list(gold_order)),
        "pair_accuracy": float(pair_accuracy),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
    }


def build_candidate_record(active_model, row, image_root, has_gold=True, top_k=CANDIDATE_TOP_K, alpha=STRUCTURED_ALPHA, beta=STRUCTURED_BETA, gamma=STRUCTURED_GAMMA):
    first_probs = score_digit_candidates(active_model, make_original_eval_example(row, "first", image_root), [1, 2, 3, 4])
    last_probs = score_digit_candidates(active_model, make_original_eval_example(row, "last", image_root), [1, 2, 3, 4])
    pair_probs = {}
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        probs = score_digit_candidates(active_model, make_original_eval_example(row, "pairwise", image_root, pair=(a, b)), [1, 2])
        pair_probs[f"{a}>{b}"] = float(probs[1])
        pair_probs[f"{b}>{a}"] = float(1.0 - probs[1])

    scored = []
    for order in PERMUTATIONS:
        score = structured_score_from_probs(
            {str(k): v for k, v in first_probs.items()},
            {str(k): v for k, v in last_probs.items()},
            pair_probs,
            order,
            alpha,
            beta,
            gamma,
        )
        scored.append({"order": list(order), "structured_score": score})
    scored = sorted(scored, key=lambda item: item["structured_score"], reverse=True)
    for rank, item in enumerate(scored, start=1):
        item["rank"] = rank

    gold_order = None
    if has_gold:
        answer = [int(value) for value in row["Answer_list"]]
        gold_order = order_to_sequence(answer)
        for item in scored:
            item["is_gold"] = item["order"] == gold_order
    else:
        for item in scored:
            item["is_gold"] = None

    return {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "gold_order": gold_order,
        "top_k": top_k,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "first_probs": {str(k): v for k, v in first_probs.items()},
        "last_probs": {str(k): v for k, v in last_probs.items()},
        "pair_probs": pair_probs,
        "candidate_orders": [item["order"] for item in scored[:top_k]],
        "structured_scores": [float(item["structured_score"]) for item in scored[:top_k]],
        "all_candidates": scored,
    }


def candidate_cache_path(split_name):
    return os.path.join(CACHE_DIR, f"{split_name}_candidates.jsonl")


def generate_candidate_cache(split_name, rows, image_root, has_gold=True, top_k=CANDIDATE_TOP_K, force=False):
    path = candidate_cache_path(split_name)
    if os.path.exists(path) and not force:
        print("[SKIP]", path)
        return read_jsonl(path)

    generator_model = load_adapter_model(INITIAL_ADAPTER_DIR, is_trainable=False)
    records = []
    try:
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"candidate cache: {split_name}"):
            records.append(build_candidate_record(generator_model, row, image_root, has_gold=has_gold, top_k=top_k))
            if len(records) % 100 == 0:
                write_jsonl(records, path)
        write_jsonl(records, path)
    finally:
        del generator_model
        gc.collect()
        torch.cuda.empty_cache()
    print("saved:", path, len(records))
    return records


def topk_recall(records, ks=(1, 3, 5, 10, 24)):
    rows = []
    for k in ks:
        hits = []
        for record in records:
            gold = record.get("gold_order")
            if gold is None:
                continue
            candidates = [item["order"] for item in record["all_candidates"][:k]]
            hits.append(int(gold in candidates))
        rows.append({"k": k, "gold_recall": float(np.mean(hits)) if hits else np.nan})
    return pd.DataFrame(rows)


In [ ]:
# 6) Generate or load candidate caches.
# Full train cache is expensive but reusable. For a smoke test, replace training_df with training_df.head(200).
train_candidates = generate_candidate_cache("train", training_df, TRAIN_IMAGE_DIR, has_gold=True)
quick_candidates = generate_candidate_cache("quick50", quick50_df, TRAIN_IMAGE_DIR, has_gold=True)
tuning_candidates = generate_candidate_cache("tuning150", tuning150_df, TRAIN_IMAGE_DIR, has_gold=True)
holdout_candidates = generate_candidate_cache("holdout150", holdout150_df, TRAIN_IMAGE_DIR, has_gold=True)

recall_tables = []
for split_name, records in [
    ("train", train_candidates),
    ("quick50", quick_candidates),
    ("tuning150", tuning_candidates),
    ("holdout150", holdout_candidates),
]:
    df = topk_recall(records)
    df.insert(0, "split", split_name)
    recall_tables.append(df)
recall_df = pd.concat(recall_tables, ignore_index=True)
recall_df.to_csv(os.path.join(EVAL_DIR, "topk_recall.csv"), index=False)
display(recall_df)


In [ ]:
# 7) Hard negative generation and A/B ranking records
def choose_negative(record, negative_type, rng):
    gold = list(record["gold_order"])
    all_orders = [item["order"] for item in record["all_candidates"]]
    wrong = [order for order in all_orders if order != gold]
    if not wrong:
        return None

    if negative_type == "decoder_top_wrong":
        return wrong[0]
    if negative_type == "first_error":
        candidates = [order for order in wrong if order[0] != gold[0]]
        return candidates[0] if candidates else None
    if negative_type == "last_error":
        candidates = [order for order in wrong if order[-1] != gold[-1]]
        return candidates[0] if candidates else None
    if negative_type == "middle_swap":
        candidates = [order for order in wrong if order[0] == gold[0] and order[-1] == gold[-1]]
        if candidates:
            return candidates[0]
        swapped = [gold[0], gold[2], gold[1], gold[3]]
        return swapped if swapped != gold else None
    if negative_type == "random":
        return list(wrong[int(rng.integers(0, len(wrong)))])
    raise ValueError(negative_type)


def structured_score_lookup(record, order):
    key = tuple(order)
    for item in record["all_candidates"]:
        if tuple(item["order"]) == key:
            return float(item["structured_score"]), int(item["rank"])
    return None, None


def make_ab_record(record, negative_order, negative_type, rng):
    gold = list(record["gold_order"])
    if negative_order is None or list(negative_order) == gold:
        return None
    if rng.random() < 0.5:
        candidate_a, candidate_b, target = gold, list(negative_order), "A"
    else:
        candidate_a, candidate_b, target = list(negative_order), gold, "B"
    gold_score, gold_rank = structured_score_lookup(record, gold)
    neg_score, neg_rank = structured_score_lookup(record, list(negative_order))
    return {
        "sample_id": record["sample_id"],
        "sentence": record["sentence"],
        "gold_order": gold,
        "candidate_a": candidate_a,
        "candidate_b": candidate_b,
        "target": target,
        "negative_order": list(negative_order),
        "negative_type": negative_type,
        "gold_rank": gold_rank,
        "negative_rank": neg_rank,
        "gold_structured_score": gold_score,
        "negative_structured_score": neg_score,
    }


def rebalance_targets(records, seed):
    rng = np.random.default_rng(seed)
    balanced = []
    for record in records:
        record = dict(record)
        if rng.random() < 0.5:
            record["candidate_a"], record["candidate_b"] = record["candidate_b"], record["candidate_a"]
            record["target"] = "A" if record["target"] == "B" else "B"
        balanced.append(record)

    a_records = [record for record in balanced if record["target"] == "A"]
    b_records = [record for record in balanced if record["target"] == "B"]
    target = min(len(a_records), len(b_records))
    if target == 0:
        raise ValueError("Cannot balance A/B targets.")
    rng.shuffle(a_records)
    rng.shuffle(b_records)
    balanced = a_records[:target] + b_records[:target]
    rng.shuffle(balanced)
    return balanced


def build_reranker_records(candidate_records, split_name):
    rng = np.random.default_rng(SEED + sum(ord(ch) for ch in split_name))
    records = []
    priority = {name: idx for idx, name in enumerate(NEGATIVE_RATIOS)}
    for record in candidate_records:
        made = []
        for negative_type in NEGATIVE_RATIOS:
            negative = choose_negative(record, negative_type, rng)
            ab = make_ab_record(record, negative, negative_type, rng)
            if ab is not None:
                made.append(ab)
        made = sorted(made, key=lambda item: priority[item["negative_type"]])[:MAX_RECORDS_PER_SAMPLE]
        records.extend(made)

    records = rebalance_targets(records, SEED + 1000 + sum(ord(ch) for ch in split_name))
    target_counts = pd.Series([record["target"] for record in records]).value_counts().to_dict()
    total = max(1, len(records))
    target_a_rate = target_counts.get("A", 0) / total
    target_b_rate = target_counts.get("B", 0) / total
    print(split_name, "records:", len(records), "target A/B:", target_counts, target_a_rate, target_b_rate)
    assert abs(target_a_rate - 0.5) < 0.02, target_counts
    assert abs(target_b_rate - 0.5) < 0.02, target_counts
    return records


def save_reranker_records(split_name, records):
    path = os.path.join(RECORD_DIR, f"{split_name}_reranker_records.jsonl")
    write_jsonl(records, path)
    distribution = {
        "split": split_name,
        "num_records": len(records),
        "target_distribution": pd.Series([record["target"] for record in records]).value_counts().to_dict(),
        "negative_type_distribution": pd.Series([record["negative_type"] for record in records]).value_counts().to_dict(),
    }
    save_json(distribution, os.path.join(RECORD_DIR, f"{split_name}_distribution.json"))
    return path


train_reranker_records = build_reranker_records(train_candidates, "train")
quick_reranker_records = build_reranker_records(quick_candidates, "quick50")
tuning_reranker_records = build_reranker_records(tuning_candidates, "tuning150")
holdout_reranker_records = build_reranker_records(holdout_candidates, "holdout150")

for split_name, records in [
    ("train", train_reranker_records),
    ("quick50", quick_reranker_records),
    ("tuning150", tuning_reranker_records),
    ("holdout150", holdout_reranker_records),
]:
    print(split_name, save_reranker_records(split_name, records))


In [ ]:
# 8) Reranker prompt, dataset, collator
TRAIN_ROW_BY_ID = train_df.set_index("Id")
TEST_ROW_BY_ID = test_df.set_index("Id")


def row_for_sample(sample_id, image_root):
    if image_root == TEST_IMAGE_DIR:
        return TEST_ROW_BY_ID.loc[str(sample_id)]
    return TRAIN_ROW_BY_ID.loc[str(sample_id)]


def reranker_instruction(example):
    return (
        "The four images are shuffled frames from one video.\n\n"
        f"Caption:\n{example['sentence']}\n\n"
        f"Candidate A:\n{format_order(example['candidate_a'])}\n\n"
        f"Candidate B:\n{format_order(example['candidate_b'])}\n\n"
        "Which candidate better represents the complete chronological order of the video?\n\n"
        "Compare the full temporal progression across all four scenes.\n"
        "Use the visual state changes, actions, objects, and the caption context.\n\n"
        "Answer only A or B."
    )


def reranker_messages(example, image_root, include_answer=False):
    sample_id = str(example["sample_id"])
    row = row_for_sample(sample_id, image_root)
    image_paths = [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]
    content = []
    for idx, _ in enumerate(image_paths, start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + reranker_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages, image_paths


class RerankerDataset(Dataset):
    def __init__(self, records, image_root):
        self.records = list(records)
        self.image_root = image_root

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        item = dict(self.records[index])
        item["image_root"] = self.image_root
        return item


class RerankerCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids, target):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            decoded_tail = self.tokenizer.decode(ids[-160:], skip_special_tokens=False)
            raise ValueError(f"Assistant prefix not found. target={target!r}, decoded_tail={decoded_tail!r}")
        labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        targets = []
        negative_types = []
        for example in batch:
            messages, image_paths = reranker_messages(example, example["image_root"], include_answer=True)
            texts.append(self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
            images.append([load_rgb(path) for path in image_paths])
            targets.append(str(example["target"]))
            negative_types.append(example.get("negative_type", "unknown"))
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.stack([self._mask_prompt(row, target) for row, target in zip(encoded["input_ids"], targets)])
        decoded_targets = [self.tokenizer.decode(row[row.ne(-100)], skip_special_tokens=True).strip() for row in labels]
        for decoded, target in zip(decoded_targets, targets):
            if target not in decoded:
                raise ValueError(f"Masked label does not contain target. target={target!r}, decoded={decoded!r}")
        encoded["labels"] = labels
        encoded["negative_type"] = negative_types
        return encoded


_mask_check = RerankerCollator(processor)([dict(record, image_root=TRAIN_IMAGE_DIR) for record in train_reranker_records[:8]])
print("reranker label mask token counts:", _mask_check["labels"].ne(-100).sum(dim=1).tolist())
del _mask_check


In [ ]:
# 9) Train ranking-only reranker from the existing best adapter
class RerankerTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        negative_types = inputs.pop("negative_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        token_losses = torch.nn.functional.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="none",
            ignore_index=-100,
        ).view_as(shift_labels)
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        loss = sample_losses.mean()
        if negative_types:
            logs = {}
            for negative_type in sorted(set(negative_types)):
                type_mask = torch.tensor([value == negative_type for value in negative_types], device=sample_losses.device)
                if type_mask.any():
                    logs[f"train_{negative_type}_loss"] = sample_losses[type_mask].mean().detach().float().item()
            if logs:
                self.log(logs)
        return (loss, outputs) if return_outputs else loss


model = load_adapter_model(INITIAL_ADAPTER_DIR, is_trainable=True)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=None,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
)

trainer = RerankerTrainer(
    model=model,
    args=training_args,
    train_dataset=RerankerDataset(train_reranker_records, TRAIN_IMAGE_DIR),
    data_collator=RerankerCollator(processor),
)
trainer.train()
final_adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)
processor.save_pretrained(OUTPUT_DIR)
print("saved:", final_adapter_dir)

del model
try:
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 10) Reranker scoring and tournament evaluation
def list_adapter_checkpoints():
    checkpoints = []
    for name in os.listdir(OUTPUT_DIR):
        path = os.path.join(OUTPUT_DIR, name)
        if name.startswith("checkpoint-") and os.path.exists(os.path.join(path, "adapter_config.json")):
            checkpoints.append(path)
    final_adapter = os.path.join(OUTPUT_DIR, "final_adapter")
    if os.path.exists(os.path.join(final_adapter, "adapter_config.json")):
        checkpoints.append(final_adapter)

    def step_key(path):
        match = re.search(r"checkpoint-(\d+)", os.path.basename(path))
        return int(match.group(1)) if match else 10**12

    return sorted(checkpoints, key=step_key)


def checkpoint_name(path):
    return os.path.basename(os.path.normpath(path))


def make_eval_pair_example(candidate_record, order_a, order_b, target="A"):
    return {
        "sample_id": candidate_record["sample_id"],
        "sentence": candidate_record["sentence"],
        "candidate_a": list(order_a),
        "candidate_b": list(order_b),
        "target": target,
    }


@torch.no_grad()
def score_ab_candidates(active_model, example, image_root):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "right"
        messages, image_paths = reranker_messages(example, image_root, include_answer=False)
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in image_paths]
        inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
        logits = outputs.logits[0, last_pos]
        token_ids = [AB_TOKEN_IDS["A"], AB_TOKEN_IDS["B"]]
        probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
        return {"A": float(probs[0]), "B": float(probs[1])}
    finally:
        processor.tokenizer.padding_side = old_padding_side


def normalize_scores(values):
    arr = np.array(values, dtype=np.float64)
    if len(arr) == 0 or float(arr.max() - arr.min()) < 1e-12:
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def rerank_candidates(active_model, candidate_record, image_root, top_k=5, lambda_value=0.0, bidirectional=True):
    orders = candidate_record["candidate_orders"][:top_k]
    structured = candidate_record["structured_scores"][:top_k]
    n = len(orders)
    wins = np.zeros(n, dtype=np.float64)
    counts = np.zeros(n, dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            probs_ab = score_ab_candidates(active_model, make_eval_pair_example(candidate_record, orders[i], orders[j]), image_root)
            p_i = probs_ab["A"]
            if bidirectional:
                probs_ba = score_ab_candidates(active_model, make_eval_pair_example(candidate_record, orders[j], orders[i]), image_root)
                p_i = 0.5 * (p_i + probs_ba["B"])
            wins[i] += p_i
            wins[j] += 1.0 - p_i
            counts[i] += 1.0
            counts[j] += 1.0
    reranker_scores = wins / np.maximum(counts, 1.0)
    structured_norm = normalize_scores(structured)
    final_scores = lambda_value * structured_norm + (1.0 - lambda_value) * reranker_scores
    best_index = int(np.argmax(final_scores))
    return {
        "pred_order": orders[best_index],
        "reranker_scores": reranker_scores.tolist(),
        "structured_norm": structured_norm.tolist(),
        "final_scores": final_scores.tolist(),
        "top_k_orders": orders,
    }


def evaluate_candidate_records(active_model, candidate_records, image_root, top_k=5, lambda_value=0.0, bidirectional=True, limit=None):
    rows = []
    records = candidate_records[:limit] if limit is not None else candidate_records
    for record in tqdm(records, desc=f"eval top{top_k} lambda{lambda_value}"):
        gold = record.get("gold_order")
        if gold is None:
            continue
        structured_pred = record["candidate_orders"][0]
        reranked = rerank_candidates(active_model, record, image_root, top_k=top_k, lambda_value=lambda_value, bidirectional=bidirectional)
        metric = order_metric_row(reranked["pred_order"], gold)
        structured_metric = order_metric_row(structured_pred, gold)
        rows.append({
            "sample_id": record["sample_id"],
            "gold_order": gold,
            "structured_pred": structured_pred,
            "pred_order": reranked["pred_order"],
            "top_k": top_k,
            "lambda": lambda_value,
            **metric,
            "structured_exact_match": structured_metric["exact_match"],
            "structured_pair_accuracy": structured_metric["pair_accuracy"],
            "structured_position_accuracy": structured_metric["position_accuracy"],
            "corrected_error": float(structured_metric["exact_match"] == 0.0 and metric["exact_match"] == 1.0),
            "introduced_error": float(structured_metric["exact_match"] == 1.0 and metric["exact_match"] == 0.0),
        })
    df = pd.DataFrame(rows)
    summary = {
        "top_k": top_k,
        "lambda": lambda_value,
        "exact_match": df["exact_match"].mean(),
        "pair_accuracy": df["pair_accuracy"].mean(),
        "position_accuracy": df["position_accuracy"].mean(),
        "structured_exact_match": df["structured_exact_match"].mean(),
        "structured_pair_accuracy": df["structured_pair_accuracy"].mean(),
        "corrected_errors": int(df["corrected_error"].sum()),
        "introduced_errors": int(df["introduced_error"].sum()),
        "net_corrections": int(df["corrected_error"].sum() - df["introduced_error"].sum()),
    }
    return summary, df


In [ ]:
# 11) Quick checkpoint evaluation
checkpoint_dirs = list_adapter_checkpoints()
print("checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])

quick_rows = []
for checkpoint_dir in checkpoint_dirs:
    ckpt = checkpoint_name(checkpoint_dir)
    eval_model = load_adapter_model(checkpoint_dir, is_trainable=False)
    try:
        for top_k in [5]:
            for lambda_value in [0.0, 0.25, 0.5]:
                summary, _ = evaluate_candidate_records(eval_model, quick_candidates, TRAIN_IMAGE_DIR, top_k=top_k, lambda_value=lambda_value, bidirectional=True)
                summary.update({"checkpoint": ckpt, "split": "quick50"})
                quick_rows.append(summary)
                pd.DataFrame(quick_rows).to_csv(os.path.join(EVAL_DIR, "checkpoint_ranking_metrics_quick.csv"), index=False)
    finally:
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()

quick_df = pd.DataFrame(quick_rows).sort_values(["exact_match", "net_corrections", "pair_accuracy"], ascending=False).reset_index(drop=True)
display(quick_df)

top_checkpoints = quick_df.head(3)["checkpoint"].drop_duplicates().tolist()
name_to_dir = {checkpoint_name(path): path for path in checkpoint_dirs}
print("top checkpoints:", top_checkpoints)


In [ ]:
# 12) Select top_k/lambda on tuning150, report once on holdout150
tuning_rows = []
for ckpt in top_checkpoints:
    checkpoint_dir = name_to_dir[ckpt]
    eval_model = load_adapter_model(checkpoint_dir, is_trainable=False)
    try:
        for top_k in RERANK_TOP_K_GRID:
            for lambda_value in LAMBDAS:
                summary, _ = evaluate_candidate_records(eval_model, tuning_candidates, TRAIN_IMAGE_DIR, top_k=top_k, lambda_value=lambda_value, bidirectional=True)
                summary.update({"checkpoint": ckpt, "split": "tuning150"})
                tuning_rows.append(summary)
                pd.DataFrame(tuning_rows).to_csv(os.path.join(EVAL_DIR, "reranker_grid_tuning.csv"), index=False)
    finally:
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()

tuning_df = pd.DataFrame(tuning_rows).sort_values(["exact_match", "net_corrections", "pair_accuracy"], ascending=False).reset_index(drop=True)
display(tuning_df)
best = tuning_df.iloc[0].to_dict()
print("BEST TUNING:", best)

best_ckpt = best["checkpoint"]
best_checkpoint_dir = name_to_dir[best_ckpt]
best_top_k = int(best["top_k"])
best_lambda = float(best["lambda"])

eval_model = load_adapter_model(best_checkpoint_dir, is_trainable=False)
try:
    holdout_summary, holdout_predictions = evaluate_candidate_records(eval_model, holdout_candidates, TRAIN_IMAGE_DIR, top_k=best_top_k, lambda_value=best_lambda, bidirectional=True)
    holdout_summary.update({"checkpoint": best_ckpt, "split": "holdout150"})
    holdout_predictions.to_csv(os.path.join(EVAL_DIR, "holdout_predictions.csv"), index=False)
    pd.DataFrame([holdout_summary]).to_csv(os.path.join(EVAL_DIR, "holdout_metrics.csv"), index=False)
finally:
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame([holdout_summary]))

if not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    shutil.copytree(best_checkpoint_dir, BEST_ADAPTER_DIR, dirs_exist_ok=True)
processor.save_pretrained(BEST_ADAPTER_DIR)

best_config = {
    "experiment": "hard_negative_permutation_reranker_v1",
    "initial_adapter": INITIAL_ADAPTER_DIR,
    "candidate_generator": {
        "type": "pair_first_last_structured_decoder",
        "top_k": CANDIDATE_TOP_K,
        "alpha": STRUCTURED_ALPHA,
        "beta": STRUCTURED_BETA,
        "gamma": STRUCTURED_GAMMA,
    },
    "reranker": {
        "type": "ab_pairwise_permutation_ranking",
        "checkpoint": best_ckpt,
        "checkpoint_dir": best_checkpoint_dir,
        "top_k": best_top_k,
        "lambda": best_lambda,
        "bidirectional": True,
    },
    "negative_ratios": NEGATIVE_RATIOS,
    "tuning": best,
    "holdout": holdout_summary,
    "baseline_test_score": 0.56544,
}
save_json(best_config, os.path.join(BEST_ADAPTER_DIR, "best_config.json"))
save_json(best_config, os.path.join(OUTPUT_DIR, "best_config.json"))
print(json.dumps(best_config, ensure_ascii=False, indent=2))


In [ ]:
# 13) Optional test inference and submission
test_candidates = generate_candidate_cache("test", test_df, TEST_IMAGE_DIR, has_gold=False)
with open(os.path.join(OUTPUT_DIR, "best_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)

test_top_k = int(best_config["reranker"]["top_k"])
test_lambda = float(best_config["reranker"]["lambda"])
test_checkpoint_dir = best_config["reranker"]["checkpoint_dir"]

test_model = load_adapter_model(test_checkpoint_dir, is_trainable=False)
submission_rows = []
test_prediction_rows = []
try:
    for record in tqdm(test_candidates, desc="test rerank"):
        reranked = rerank_candidates(test_model, record, TEST_IMAGE_DIR, top_k=test_top_k, lambda_value=test_lambda, bidirectional=True)
        pred_order = reranked["pred_order"]
        submission_rows.append({"Id": record["sample_id"], "Answer": str(sequence_to_answer(pred_order))})
        test_prediction_rows.append({
            "sample_id": record["sample_id"],
            "pred_order": pred_order,
            "candidate_orders": reranked["top_k_orders"],
            "final_scores": reranked["final_scores"],
            "reranker_scores": reranked["reranker_scores"],
        })
finally:
    del test_model
    gc.collect()
    torch.cuda.empty_cache()

submission = pd.DataFrame(submission_rows)
if sample_submission_df is not None:
    sample_ids = sample_submission_df["Id"].astype(str).tolist()
    submission["Id"] = submission["Id"].astype(str)
    submission = submission.set_index("Id").loc[sample_ids].reset_index()
    assert len(submission) == len(sample_submission_df), "submission row count mismatch"
    assert submission["Id"].astype(str).tolist() == sample_ids, "submission Id order mismatch"
submission.to_csv(SUBMIT_PATH, index=False)
write_jsonl(test_prediction_rows, os.path.join(EVAL_DIR, "test_reranker_predictions.jsonl"))
shutil.copy2(SUBMIT_PATH, os.path.join(BEST_ADAPTER_DIR, "submission.csv"))
display(submission.head())
print("submission saved:", SUBMIT_PATH)


In [ ]:
# 14) Run config and artifact summary
run_config = {
    "experiment": "hard_negative_permutation_reranker_v1",
    "run_id": RUN_ID,
    "initial_adapter": INITIAL_ADAPTER_DIR,
    "model_repo_id": MODEL_REPO_ID,
    "model_id": MODEL_ID,
    "split_dir": SPLIT_DIR,
    "split_hashes": {name: ids_hash(ids) for name, ids in split_ids.items()},
    "candidate_top_k": CANDIDATE_TOP_K,
    "structured_weights": {"alpha": STRUCTURED_ALPHA, "beta": STRUCTURED_BETA, "gamma": STRUCTURED_GAMMA},
    "negative_ratios": NEGATIVE_RATIOS,
    "max_records_per_sample": MAX_RECORDS_PER_SAMPLE,
    "learning_rate": LEARNING_RATE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "baseline_test_score": 0.56544,
    "output_dir": OUTPUT_DIR,
}
save_json(run_config, os.path.join(RUN_ROOT, "run_config.json"))
save_json(run_config, os.path.join(OUTPUT_DIR, "run_config.json"))
print("Artifacts")
print("run_config:", os.path.join(OUTPUT_DIR, "run_config.json"))
print("candidate cache:", CACHE_DIR)
print("reranker records:", RECORD_DIR)
print("eval:", EVAL_DIR)
print("best adapter:", BEST_ADAPTER_DIR)
print("submission:", SUBMIT_PATH)


## Notes

- The existing structured decoder is kept. The reranker only chooses among candidates generated by that decoder.
- The train/validation split is loaded from the Qwen2 best-run split directory when present.
- A/B targets are balanced to 50:50 within 2%.
- Tuning and holdout are separated: tuning chooses checkpoint/top-K/lambda, holdout reports that fixed choice.
- Candidate caches are saved as JSONL so expensive Qwen2 probability extraction can be reused.
